In [ ]:
# Kurulum: Colab'da repoyu indirir, yerelde hiçbir şey yapmaz.
import os, sys
REPO = "https://github.com/mehmetalibeylikci/python-cevresel-modelleme"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/python-cevresel-modelleme"):
        !git clone -q {REPO} /content/python-cevresel-modelleme
    %cd /content/python-cevresel-modelleme
    %pip install -q sweetviz
else:
    while not os.path.exists("requirements.txt") and os.getcwd() != "/":
        os.chdir("..")
sys.path.insert(0, os.getcwd())

# Gün 1 — Veri Hazırlama

**İYTE Çevre Topluluğu · Python ile Çevresel Modelleme**

Bugün, Tahtalı Barajı modelimizi besleyecek meteoroloji verisini **okuyacak, inceleyecek, temizleyecek** ve **aylık** hâle getireceğiz.

**Öğrenme hedefleri**
- pandas ile bir CSV dosyasını okumak ve hızlıca tanımak
- Gerçek hayattaki tipik veri sorunlarını bulmak ve düzeltmek
- Günlük veriyi aylığa çevirmek (`resample`)
- Temiz veriyi bir dosyaya kaydetmek

| Süre | Konu |
|---|---|
| 0–15 | Problem: İzmir su krizi ve Tahtalı, harita, Colab turu |
| 15–45 | pandas temelleri |
| 45–55 | Sweetviz ile otomatik keşif raporu |
| 55–90 | Veri temizliği |
| 90–105 | Günlükten aylığa |
| 105–120 | İlk grafik, alıştırma ve dosyayı kaydetme |

## 1. Problem: İzmir'in suyu nereden geliyor?

Tahtalı Barajı, İzmir kent merkezine içme suyu sağlayan **en büyük kaynaktır.** 2025 yazında baraj neredeyse tamamen boşaldı: 11 Ağustos 2025'te aktif doluluk yalnızca **%7,45** idi. 2026'da yağışlarla toparlandı ve 14 Ekim 2026'da **%42,75** oldu.

Üç gün boyunca şu soruyu soracağız: **Barajın doluluğu yağışa, buharlaşmaya ve şehrin su çekimine nasıl bağlı?**

Bunun için önce barajın nerede olduğuna bakalım. İZSU'nun açık veri dosyasında barajların koordinatları var:

In [ ]:
import pandas as pd

dams = pd.read_csv("data/raw/izsu_baraj_2026-10-14.csv", sep=";", encoding="utf-8-sig")
dams[["BARAJ_KUYU_ADI", "DOLULUK_ORANI", "ENLEM", "BOYLAM"]]

In [ ]:
import folium

tahtali = dams[dams["BARAJ_KUYU_ADI"] == "Tahtalı Barajı"].iloc[0]
dam_map = folium.Map(location=[38.30, 27.15], zoom_start=9)
for _, dam in dams.iterrows():
    folium.Marker(
        [dam["ENLEM"], dam["BOYLAM"]],
        popup=f"{dam['BARAJ_KUYU_ADI']}: %{dam['DOLULUK_ORANI']}",
        icon=folium.Icon(color="red" if dam["BARAJ_KUYU_ADI"] == "Tahtalı Barajı" else "blue"),
    ).add_to(dam_map)
dam_map

### Kısa bir terim sözlüğü

Kodda değişken isimlerini İngilizce yazıyoruz. Python ve kütüphanelerinin belgeleri İngilizce olduğu için bu iyi bir alışkanlık.

| Türkçe | Kodda | Açıklama |
|---|---|---|
| yağış | `precipitation` | gökyüzünden düşen su (mm) |
| buharlaşma potansiyeli | `et0` | referans evapotranspirasyon (mm) |
| depolama | `storage` | göldeki su hacmi (hm³ = milyon m³) |
| akış katsayısı | `runoff_coef` | havzaya düşen yağmurun barajda toplanan oranı |
| çekim | `withdrawal` | şehre verilen su |
| taşma | `spill` | baraj dolunca dolusavaktan çıkan su |
| aylık toplam | `resample(...).sum()` | günlük değerleri ay ay toplamak |

## 2. pandas temelleri

**pandas**, tablo şeklindeki veriyle çalışmanın standart yoludur. Bir tabloya `DataFrame`, tek bir sütuna `Series` denir.

Verimiz: Tahtalı noktası için 2014–2026 arası **günlük yağış ve ET0** (ERA5 reanaliz verisi, Open-Meteo). Ama dikkat: bu dosya, eğitim için **kasıtlı olarak bozulmuş** bir kopya. Gerçek hayattaki veriler de genelde böyle gelir!

In [ ]:
raw = pd.read_csv("data/raw/tahtali_era5_daily_kirli.csv")
raw.head()

In [ ]:
raw.info()

`info()` bize üç şey söylüyor:
1. Kaç satır var?
2. Hangi sütunlar var?
3. Her sütunun **türü** (`dtype`) ne?

`date` sütununun türü `object` (pandas 3'te `str`), yani pandas onu tarih değil **metin** olarak görüyor. Bunu düzeltmemiz gerekecek.

`describe()` ise sayısal sütunların özet istatistiklerini verir:

In [ ]:
raw.describe()

🤔 **Tartışalım:** `describe()` çıktısında tuhaf görünen bir şey var mı? Yağışın en küçük (`min`) ve en büyük (`max`) değerlerine bakın.

Bir sütunu seçmek, filtrelemek ve yeni sütun eklemek pandas'ın temel hareketleridir:

In [ ]:
precip = raw["precipitation_mm"]            # tek sütun → Series
print(precip.mean())

rainy_days = raw[raw["precipitation_mm"] > 50]   # koşula göre satır filtreleme
print(len(rainy_days), "gün 50 mm'den fazla yağış almış")

### 🟢 Alıştırma 1
ET0'ın (`et0_mm`) **ortalamasını** ve **en büyük değerini** yazdırın. Sonra ET0'ın 8 mm'den büyük olduğu kaç gün var, bulun.

In [ ]:
# TODO: kendin yaz
...

## 3. Sweetviz ile kuşbakışı

Veriyi elle incelemeden önce, otomatik bir **keşifsel analiz raporu** ile hızlıca bakmak çok işe yarar. Sweetviz tek satırla her sütunun dağılımını, eksik değerleri ve ilişkileri gösteren bir rapor üretir.

In [ ]:
import logging

import matplotlib.pyplot as plt
import sweetviz

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)  # gereksiz font uyarılarını gizle

report = sweetviz.analyze(raw)
report.show_html("sweetviz_rapor.html", open_browser=False)
# Colab'da raporu notebook içinde görmek için şu satırı kullanın:
# report.show_notebook()

plt.rcdefaults()  # Sweetviz grafik ayarlarını değiştirir; varsayılanlara dönüyoruz

Rapor bize ipuçları verir, ama **sorunları çözmez.** Raporda gördüğümüz sorunları şimdi tek tek bulup düzelteceğiz. Rapordan neler dikkatinizi çekti?

## 4. Veri temizliği

### 4.1 Tarihler

Tarih sütununu gerçek tarihe çevirmeyi deneyelim:

In [ ]:
try:
    pd.to_datetime(raw["date"], format="%Y-%m-%d")
except ValueError as error:
    print("Hata:", error)

Bazı satırlarda tarih `05.03.2019` biçiminde yazılmış! İki biçimi ayrı ayrı çevirip birleştirelim. `errors="coerce"`, çevrilemeyen değerleri hata yerine `NaT` (boş tarih) yapar.

In [ ]:
df = raw.copy()
iso = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce")
dotted = pd.to_datetime(df["date"], format="%d.%m.%Y", errors="coerce")
print("Noktalı biçimdeki satır sayısı:", dotted.notna().sum())

df["date"] = iso.fillna(dotted)
print("Çevrilemeyen tarih kaldı mı?", df["date"].isna().sum())

### 4.2 Tekrarlanan satırlar

In [ ]:
print("Tekrarlanan satır:", df.duplicated().sum())
df = df.drop_duplicates()
print("Aynı tarih birden fazla kez geçiyor mu?", df["date"].duplicated().sum())

Artık her tarih tek bir kez geçiyor. Tarihi **indeks** yapalım: pandas'ın zaman serisi araçları bunu bekler.

In [ ]:
df = df.set_index("date").sort_index()
df.head()

### 4.3 Fiziksel olarak imkânsız değerler

Yağış **negatif** olamaz. İzmir'de bir günde 900 mm yağış da (yıllık yağışın neredeyse tamamı!) gerçekçi değil. Bu değerleri bulalım:

In [ ]:
impossible = (df["precipitation_mm"] < 0) | (df["precipitation_mm"] > 500)
df[impossible]

Bu değerlerin gerçekte ne olduğunu bilemeyiz. Bu yüzden onları silmek yerine **eksik** (`NaN`) olarak işaretleyelim. Birazdan diğer eksiklerle birlikte dolduracağız.

In [ ]:
import numpy as np

df.loc[impossible, "precipitation_mm"] = np.nan

### 4.4 Birim hatası

Bu en sinsi hatadır: değerler "makul görünür" ama yanlıştır. Yıllık toplam yağışa bakalım:

In [ ]:
import matplotlib.pyplot as plt

annual = df["precipitation_mm"].groupby(df.index.year).sum()
ax = annual.plot.bar(color="steelblue", figsize=(9, 3.5))
ax.set_title("Yıllık toplam yağış")
ax.set_xlabel("Yıl")
ax.set_ylabel("Yağış (mm)")
plt.show()

### 🟢 Alıştırma 2
Grafikte bir yıl diğerlerinden çok farklı. Hangi yıl? O yılın toplamı, diğer yılların ortalamasının yaklaşık kaçta kaçı? Bu oran bize hatanın ne olduğunu söylüyor olabilir.

In [ ]:
# TODO: kendin yaz
...

Oran yaklaşık **1/10**. Muhtemelen o yıl yağış **santimetre** cinsinden kaydedilmiş. Düzeltelim:

In [ ]:
df.loc["2019", "precipitation_mm"] = df.loc["2019", "precipitation_mm"] * 10
df["precipitation_mm"].groupby(df.index.year).sum().round()

> 💡 **Gerçek hayatta** böyle bir düzeltmeyi yapmadan önce veri sağlayıcısının belgelerine bakar ya da onlarla iletişime geçeriz. Burada hatayı bilerek biz koyduk.

### 4.5 Eksik günler

2014-01-01'den 2026-09-30'a kadar her gün olmalı. Kaç gün eksik?

In [ ]:
full_range = pd.date_range("2014-01-01", "2026-09-30", freq="D")
print("Olması gereken gün:", len(full_range))
print("Elimizdeki gün:", len(df))

df = df.reindex(full_range)
df.index.name = "date"
print("Eksik değerler:\n", df.isna().sum())

`reindex` eksik günleri tabloya `NaN` olarak ekledi. Şimdi bunları dolduracağız. Seçenekler:

| Yöntem | Ne yapar | Ne zaman uygun |
|---|---|---|
| `fillna(0)` | eksik = 0 | "ölçüm yoksa olay da yoktur" dediğimiz durumlar |
| `ffill()` | bir önceki günün değeri | yavaş değişen büyüklükler |
| `interpolate(method="time")` | komşu günler arasında doğrusal geçiş | düzgün değişen büyüklükler (sıcaklık, ET0) |

ET0 günden güne yavaş değişir, bu yüzden interpolasyon çok iyi çalışır. Yağış ise ani bir olaydır: eksik bir günde yağmur yağıp yağmadığını gerçekten bilemeyiz. Biz her ikisi için de basit ve savunulabilir bir seçim olarak **interpolasyon** kullanacağız. Sonra bu seçimin ne kadar hata yaptığını ölçeceğiz.

In [ ]:
df = df.interpolate(method="time")
print("Kalan eksik:", df.isna().sum().sum())

## 5. Günlükten aylığa: `resample`

Modelimiz **aylık** çalışacak. `resample("MS")` veriyi ay başlarına göre gruplar (*Month Start*). Sonra her grubu nasıl özetleyeceğimizi söyleriz:

- **Yağış ve ET0** bir **miktardır** (mm). Ayın toplam miktarı için **`sum()`** kullanırız.
- **Sıcaklık** gibi bir **durum** büyüklüğü olsaydı **`mean()`** kullanırdık. Bir ayın sıcaklıklarını toplamak anlamsızdır!

In [ ]:
monthly = df.resample("MS").sum()
monthly.head()

### 🟢 Alıştırma 3
Tüm dönemin **en yağışlı ayı** hangisi, ve o ay kaç mm yağış düşmüş?

In [ ]:
# TODO: kendin yaz
...

## 6. İlk grafik ve kaydetme

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.bar(monthly.index, monthly["precipitation_mm"], width=20, color="steelblue", label="Yağış")
ax.plot(monthly.index, monthly["et0_mm"], color="darkorange", label="ET0 (buharlaşma potansiyeli)")
ax.set_title("Tahtalı — aylık yağış ve ET0")
ax.set_ylabel("mm/ay")
ax.legend()
plt.show()

🤔 **Tartışalım:** Yağış hangi aylarda yoğunlaşıyor, ET0 hangi aylarda? Bu, baraj için ne anlama geliyor?

### 🔵 Alıştırma 4
Her **takvim ayının** (Ocak, Şubat, ...) ortalama yağışını ve ortalama ET0'ını hesaplayın ve yan yana bir çubuk grafikte gösterin. Buna **aylık klimatoloji** denir.

*İpucu:* `monthly.groupby(monthly.index.month).mean()`

In [ ]:
# TODO: kendin yaz
...

Son olarak temiz aylık veriyi Gün 2'de kullanmak üzere kaydedelim. Ayı `YYYY-MM` biçiminde yazıyoruz:

In [ ]:
out = monthly.round(2)
out.index = out.index.strftime("%Y-%m")
out.index.name = "month"
out.to_csv("data/processed/tahtali_monthly.csv")
print(len(out), "ay kaydedildi.")

🎉 **Gün 1 tamam!** Bugün ham ve kirli bir veriyi modelin kullanabileceği temiz bir aylık seriye çevirdik.

Yarın: Bu veriyle **Tahtalı'nın su bütçesi modelini** kendimiz yazacağız.

---
## Zaman kalırsa: Temizliğimiz ne kadar iyiydi?

Bu eğitimde şanslıyız: bozulmamış orijinal dosya da elimizde (`tahtali_era5_daily.csv`). Gerçek hayatta bu lüksümüz olmaz. Kendi temizlediğimiz aylık veriyi orijinaliyle karşılaştıralım.

In [ ]:
original = pd.read_csv("data/raw/tahtali_era5_daily.csv", parse_dates=["date"], index_col="date")
original_monthly = original.resample("MS").sum()
error = monthly - original_monthly
print("Toplam yağış farkı: %", round(100 * error["precipitation_mm"].sum() / original_monthly["precipitation_mm"].sum(), 2))
print("Bir aydaki en büyük yağış farkı:", round(error["precipitation_mm"].abs().max(), 1), "mm")
print("Bir aydaki en büyük ET0 farkı:", round(error["et0_mm"].abs().max(), 1), "mm")

Toplamda hata %1'in altında, ama tek bir ayda yağış onlarca mm sapabiliyor. Neden? İpucu: eksik bir gün **yağmurlu** bir günse ne olur?

### 🔵 Alıştırma 5
Eksik yağış günlerini interpolasyon yerine `fillna(0)` ile doldurursanız toplam fark nasıl değişir? Hangisi daha iyi, neden?